In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Setting things up

In [ ]:
#This lines just clean up the text outputs in this block 0.180412
from IPython.display import clear_output
clear_output()

In [ ]:
#import the packages we are going to use
import tensorflow as tf #ML heavy lifting
import pandas as pd #data handling (tables)
import numpy as np #multi-dimensional vectors of data
from matplotlib import pyplot as plt #for plotting
import math

from tensorflow.keras import datasets, layers, models

# Generating some data

In order to compare the performance of some test algorithms, we can generate some pseudo-random data.

We will generate two independent datasets:
- a training dataset (train_df)
- a test dataset (eval_df)

In [ ]:
N_elements = 20000

def gen_class(x, y):
  if y > math.sin(10*x):
  #if y > x:
    return 1
  else:
    return 0


# Generate the random dataset
names = ['x', 'y']
x_vec = np.random.uniform(0.,1.,N_elements)
y_vec = np.random.uniform(0.,1.,N_elements)

my_truth = []
i = 0
for element in y_vec:
  my_truth.append(gen_class(x_vec[i], element))
  i = i + 1

truth_vec = np.array(my_truth)

data = np.stack((x_vec,y_vec,truth_vec), axis=1)
all_df = pd.DataFrame(data, columns=['x', 'y','truth'])

#Useful function to split a dataframe in train and test samples
from sklearn.model_selection import train_test_split
train_df_all, eval_df_all = train_test_split(all_df, test_size=0.5)

#This creates two pandas dataframes we can use to train the discriminators
train_df = train_df_all[names].copy()
train_df_labels = pd.DataFrame(train_df_all[['truth']].copy())
eval_df = eval_df_all[names].copy()
eval_df_labels = pd.DataFrame(eval_df_all[['truth']].copy())

In [ ]:
train_df.head() #inspect the dataset

In [ ]:
print(train_df.shape[0], train_df.shape[1])
# the first number is the number of
# events, the second the number of features

train_df.x.hist(bins=10)
# example plot of the data

# Model definition and training

In [ ]:
# Train a linear estimator
LIN_estimator = models.Sequential()
LIN_estimator.add(layers.Dense(1))

LIN_estimator.compile(optimizer=tf.keras.optimizers.SGD(
                                               learning_rate=0.005,
                                               momentum=0.95,
                                               nesterov=True
                                           ),
                      loss=tf.keras.losses.MeanSquaredError(),
                      metrics=['accuracy'])

LIN_training_history = LIN_estimator.fit(train_df,
                            train_df_labels,
                            epochs=10,
                            # To suppress logging uncomment verbose=0,
                            validation_split = 0.2)

LIN_estimator.summary()

In [ ]:
def plot_loss(history):
  plt.plot(history.history['loss'], label='Training dataset')
  plt.plot(history.history['val_loss'], label='Validation dataset')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()
  plt.grid(True)

plot_loss(LIN_training_history)

In [ ]:
# Train a neural network with:
# 2 inputs
# 2 hidden layers, each composed by 64 neurons

DNN_estimator = models.Sequential()
DNN_estimator.add(layers.Dense(60, activation='relu'))
DNN_estimator.add(layers.Dense(60, activation='relu'))
DNN_estimator.add(layers.Dense(1, activation='sigmoid'))

DNN_estimator.compile(optimizer=tf.keras.optimizers.SGD(
                                               learning_rate=0.005,
                                               momentum=0.95,
                                               nesterov=True
                                           ),
                      loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
                      metrics=['accuracy'])

DNN_training_history = DNN_estimator.fit(train_df,
                            train_df_labels,
                            epochs=40,
                            # To suppress logging uncomment verbose=0,
                            validation_split = 0.2)

DNN_estimator.summary()

In [ ]:
plot_loss(DNN_training_history)

# Evaluation of NN performance

In [ ]:
#Let's look at the outputs of the linear and NN discriminants
#Evaluate the trained algorithms on the test dataset

list_estimators = [LIN_estimator, DNN_estimator]

#To make some instructive plots, let's split the data according to its label
eval_df_0 = eval_df_all.loc[eval_df_all['truth'] == 0].filter(['x','y'], axis=1)
eval_df_1 = eval_df_all.loc[eval_df_all['truth'] == 1].filter(['x','y'], axis=1)

fig, ax = plt.subplots(1,len(list_estimators), figsize=(5*len(list_estimators), 5))
for i, estimator in enumerate(list_estimators):
  pred_dicts_0 = list(estimator.predict(eval_df_0))
  pred_dicts_1 = list(estimator.predict(eval_df_1))
  probs_0 = pd.DataFrame(pred_dicts_0)
  probs_1 = pd.DataFrame(pred_dicts_1)

  probs_0.plot(ax=ax[i], kind='hist', bins=100, title='', alpha=0.7)
  probs_1.plot(ax=ax[i], kind='hist', bins=100, alpha=0.7)
  ax[i].legend(["0", "1"],frameon=False, prop={'size': 16})

In [ ]:
from sklearn.metrics import roc_curve

'''
Plot ROC curves!

A ROC curve is a performance measurement for a classification problem at various
thresholds settings. The ROC is a probability curve and AUC (Area Under Curve)
represents the degree or measure of separability. It tells how much your model
is capable of distinguishing between classes.
'''
pred_dicts_LIN = LIN_estimator.predict(eval_df).flatten()
pred_dicts_DNN = DNN_estimator.predict(eval_df).flatten()
probs_LIN = pd.DataFrame(pred_dicts_LIN)
probs_DNN = pd.DataFrame(pred_dicts_DNN)

fig, ax = plt.subplots()
fpr, tpr, _ = roc_curve(eval_df_labels, pred_dicts_LIN)
ax.plot(fpr, tpr, label='LIN')
fpr2, tpr2, _ = roc_curve(eval_df_labels, pred_dicts_DNN)
ax.plot(fpr2, tpr2, label='DNN')

ax.set_title('ROC curve', size='20')
ax.set_xlabel('False positive rate', size='20')
ax.set_ylabel('True positive rate', size='20')
ax.set_xlim(0,)
ax.set_ylim(0,)
ax.set_aspect('equal')

ax.legend(frameon=False, prop={'size': 16})

In [ ]:
# Let's add back the classification information in the dataframes

# Plot the truth for reference
# Plot
fig, ax1 = plt.subplots(1,3, figsize=(15,5))
eval_df_0.plot.scatter(ax=ax1[0], x='x',y='y', c='DarkBlue', title='Truth')
eval_df_1.plot.scatter(ax=ax1[0], x='x',y='y', c='DarkRed')

summary_df = eval_df.copy()
summary_df['lin'] = LIN_estimator.predict(eval_df).flatten()
summary_df['NN'] = DNN_estimator.predict(eval_df).flatten()

# Let's look at how things were classified
my_df_sig = summary_df.loc[summary_df['lin'] > 0.5]
my_df_bkg = summary_df.loc[summary_df['lin'] < 0.5]

# Plot
my_df_sig.plot.scatter(ax=ax1[1], x='x',y='y', c='DarkGreen', title='Linear')
my_df_bkg.plot.scatter(ax=ax1[1], x='x',y='y', c='Yellow')

# Let's look at how things were classified
my_df_sig = summary_df.loc[summary_df['NN'] > 0.1]
my_df_bkg = summary_df.loc[summary_df['NN'] < 0.1]

# Plot
my_df_sig.plot.scatter(ax=ax1[2], x='x',y='y', c='DarkGreen', title='DNN')
my_df_bkg.plot.scatter(ax=ax1[2], x='x',y='y', c='Yellow')

This was too easy!

# Try something more complex

For example, what about a sine function?
Go back to the start and change

```
  if y > x:
```
into
```
if y > math.sin(10*x):
```

# Up for a challenge?

Download the training data (data.csv and labels.csv)

Can you tell me what's the fraction of events of class "0" in test.csv?

In [ ]:
# Get the input training files
!wget https://www.desy.de/~fmeloni/resources/data.csv
!wget https://www.desy.de/~fmeloni/resources/labels.csv

#Get the input challenge file
!wget https://www.desy.de/~fmeloni/resources/test.csv

#To get the data from a csv into a dataframe, you can:
data = pd.read_csv('data.csv')